In [ ]:
import pandas as pd
import networkx as nx
import random
from gensim.models import Word2Vec

# -------- Step 1: Load Graph from Edge List --------
def load_graph(edge_list_path):
    G = nx.read_edgelist(edge_list_path)
    return G

# -------- Step 2: Generate Random Walks --------
def generate_random_walks(G, num_walks, walk_len):
    walks = []
    nodes = list(G.nodes())
    for _ in range(num_walks):
        random.shuffle(nodes)
        for node in nodes:
            walk = random_walk(G, node, walk_len)
            walks.append(walk)
    return walks

def random_walk(G, start_node, walk_len):
    walk = [start_node]
    while len(walk) < walk_len:
        cur = walk[-1]
        neighbors = list(G.neighbors(cur))
        if neighbors:
            next_node = random.choice(neighbors)
            walk.append(next_node)
        else:
            break
    return walk

# -------- Step 3: Train Word2Vec (DeepWalk) --------
def train_deepwalk(walks, emb_size, window, workers, epochs):
    model = Word2Vec(
        sentences=walks,
        vector_size=emb_size,
        window=window,
        min_count=0,
        sg=1,  # Skip-gram
        workers=workers,
        epochs=epochs
    )
    return model

# -------- Step 4: Save Embeddings --------
def save_embeddings(model, output_path):
    model.wv.save_word2vec_format(output_path)

In [8]:
# -------- Parameters --------
walk_length = 40
num_walks_per_node = 40
embedding_size = 128
window_size = 5
workers = 4
epochs = 5

edge_path = '/itf-fi-ml/shared/users/ziyuzh/svm/data/ppi_full_2019.txt'
out_path = '/itf-fi-ml/shared/users/ziyuzh/svm/data/ppi_full_2019_dw_emb_40_40.txt'

def deepwalk_pipeline(edge_list_path, output_path):
    G = load_graph(edge_list_path)
    walks = generate_random_walks(G, num_walks_per_node, walk_length)
    walks = [[str(node) for node in walk] for walk in walks]  # ensure string type
    model = train_deepwalk(walks, embedding_size, window_size, workers, epochs)
    save_embeddings(model, output_path)
    print(f"Embeddings saved to: {output_path}")

deepwalk_pipeline(edge_path, out_path)

KeyboardInterrupt: 

In [1]:
import pandas as pd
import mygene

def get_map_df(ensembl_ids,input_type):
    mg = mygene.MyGeneInfo()
    # Query mygene for UniProt and Entrez gene ID mappings
    results = mg.querymany(
        ensembl_ids,
        scopes=input_type,
        fields='uniprot,entrezgene',
        species='human'
    )

    results_df = pd.DataFrame(results)
    results_df = results_df[~results_df['entrezgene'].isna()]
    results_df['uniprot_ids'] = results_df['uniprot'].apply(
        lambda x: list(x.values())[0] if isinstance(x, dict) and 'Swiss-Prot' in x else None)
    results_df = results_df[~results_df['uniprot_ids'].isna()]
    results_df[results_df['uniprot_ids'].apply(lambda x: isinstance(x, list) and len(x) > 1)]
    return results_df

In [9]:
file_list = [
'/itf-fi-ml/shared/users/ziyuzh/svm/data/ppi_full_2016_dw_emb_10.txt',
'/itf-fi-ml/shared/users/ziyuzh/svm/data/ppi_full_2016_dw_emb_40.txt',
'/itf-fi-ml/shared/users/ziyuzh/svm/data/ppi_full_2016_dw_emb_80.txt',
'/itf-fi-ml/shared/users/ziyuzh/svm/data/ppi_full_2019_dw_emb_10.txt',
'/itf-fi-ml/shared/users/ziyuzh/svm/data/ppi_full_2019_dw_emb_40.txt',
'/itf-fi-ml/shared/users/ziyuzh/svm/data/ppi_full_2019_dw_emb_80.txt']

for file_path in file_list:
    ppi_emb = pd.read_csv(file_path,sep='\s+', skiprows=1, header=None)
    ppi_emb.columns = ['string_id'] + [f'feature_{i}' for i in range(1, len(ppi_emb.columns))]

    ppi_ids_map = get_map_df(ppi_emb['string_id'].str.split('.').str[1],'ensembl.protein')

    ppi_set = set()
    for values in ppi_ids_map['uniprot_ids']:
        if isinstance(values, list) and len(values) > 1:
            ppi_set.update(values)  # Add all elements in the list
        else:
            ppi_set.add(values)

    string_ids = []
    one2more = []
    more2one = []  # to collect subdfs with multiple or zero matches

    for uniport_ids in list(ppi_set):
        subdf = ppi_ids_map[ppi_ids_map['uniprot_ids'].str.contains(uniport_ids, na=False)]
        
        if len(subdf) == 1:
            if isinstance(subdf['uniprot_ids'], list) and len(values) > 1:
                one2more.append(subdf)
            else:
                string_ids.append(uniport_ids)
        else:
            more2one.append(subdf)

    more2one_df = pd.concat(more2one, ignore_index=True)

    # Prepare list to store the results
    aggregated_rows = []

    # Iterate over each UniProt ID group
    for protein_id, subdf in more2one_df.groupby('uniprot_ids'):
        # Get list of ENSP IDs
        ensp_ids = subdf['query'].tolist()

        # Add '9606.' prefix to each ENSP ID
        ensp_ids = ['9606.' + ensp_id for ensp_id in ensp_ids]

        # Select corresponding rows from ppi_emb where 'string_id' is in ensp_ids
        matched_ppi = ppi_emb[ppi_emb['string_id'].isin(ensp_ids)]

        if not matched_ppi.empty:
            # Calculate the mean of all feature columns (exclude 'string_id')
            mean_features = matched_ppi.drop(columns=['string_id']).mean()

            # Create a new row with UniProt ID and the averaged features
            mean_features['string_id'] = protein_id

            # Add to the results list
            aggregated_rows.append(mean_features)

    # Convert the list of Series into a DataFrame
    aggregated_df = pd.DataFrame(aggregated_rows)

    # Optional: Reorder columns to have 'uniprot_id' first
    cols = ['string_id'] + [col for col in aggregated_df.columns if col != 'string_id']
    aggregated_df = aggregated_df[cols]

    # Step 1: Prepare ENSP IDs with '9606.' prefix
    ensp_ids = ['9606.' + ensp_id for ensp_id in ppi_ids_map[ppi_ids_map['uniprot_ids'].isin(string_ids)]['query'].tolist()]

    # Step 2: Select matching rows from ppi_emb
    other_ppi = ppi_emb[ppi_emb['string_id'].isin(ensp_ids)].copy()

    # Step 3: Map 'string_id' back to 'uniprot_ids'
    # First, create mapping from ENSP ID with '9606.' prefix to UniProt ID
    ensp_to_uniprot = ppi_ids_map[ppi_ids_map['uniprot_ids'].isin(string_ids)].set_index('query')['uniprot_ids'].to_dict()

    # Apply mapping to refill 'string_id' with corresponding UniProt ID
    other_ppi['string_id'] = other_ppi['string_id'].apply(lambda x: ensp_to_uniprot[x.replace('9606.', '')])

    ppi_df = pd.concat([other_ppi, aggregated_df], ignore_index=True)

    ppi_df.to_csv(file_path,index = False)

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
1803 input query terms found no hit:	['ENSP00000368699', 'ENSP00000350052', 'ENSP00000349960', 'ENSP00000310146', 'ENSP00000464265', 'ENS
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
1803 input query terms found no hit:	['ENSP00000368699', 'ENSP00000350052', 'ENSP00000349960', 'ENSP00000277541', 'ENSP00000464265', 'ENS
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
1803 input query terms found no hit:	['ENSP00000350052', 'ENSP00000368699', 'ENSP00000349960', 'ENSP00000310146', 'ENSP00000464265', 'ENS
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. N

In [11]:
file_path

'/itf-fi-ml/shared/users/ziyuzh/svm/data/ppi_full_2019_dw_emb_80.txt'